### Links and Resources
| Concept | Description |
|---|---|
| [Structured Streaming guide](https://spark.apache.org/docs/latest/structured-streaming-programming-guide.html) | Streaming semantics |
| [Unsupported operations](https://spark.apache.org/docs/latest/structured-streaming-programming-guide.html#unsupported-operations) | What functions can't appear in a stream |
| [Watermarks & windows](https://spark.apache.org/docs/latest/structured-streaming-programming-guide.html#window-operations-on-event-time) | Stateful windowing |
| [`applyInPandasWithState`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.GroupedData.applyInPandasWithState.html) | Stateful Pandas UDFs in streams |

In [ ]:
import os, sys

try:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.getOrCreate()
except ImportError:
    if not os.getenv("DATABRICKS_RUNTIME_VERSION"):
        raise RuntimeError(
            f"Wrong Python kernel \u2014 'databricks.connect' not found in:\n  {sys.executable}\n\n"
            "In VSCode: click the kernel selector (top-right of notebook) "
            "and choose the interpreter at:\n"
            r"  C:\Source\Dsa\DataBricks\.venv\Scripts\python.exe"
        )
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.getOrCreate()

from pyspark.sql import functions as F


def display(df, n=20):
    df.show(n, truncate=False)

# Functions in Streaming

The `Streaming/` folder of this project covers the *mechanics* of structured streaming. This notebook is narrower: **which functions you can use inside a streaming query, which you can't, and the gotchas that don't show up in batch.**

In [ ]:
import time

events = (spark.readStream
    .format("rate")
    .option("rowsPerSecond", 20)
    .option("numPartitions", 1)
    .load()
    .withColumnRenamed("value", "event_id")
    .withColumn("user_id", (F.col("event_id") % 5).cast("int"))
    .withColumn("amount",  F.rand(seed=1) * 100))

events.printSchema()
print("isStreaming:", events.isStreaming)

In [ ]:
def peek(stream_df, name, seconds=4, limit=10):
    q = (stream_df.writeStream
         .format("memory")
         .queryName(name)
         .outputMode("append")
         .start())
    time.sleep(seconds)
    q.stop()
    return spark.sql(f"SELECT * FROM {name} LIMIT {limit}")

## Stateless functions — always fine

Anything that operates on one row at a time works identically to batch: arithmetic, string functions, date functions, `when`, `coalesce`, `from_json`, higher-order array functions, scalar UDFs (Python and Pandas), SQL UDFs.

In [ ]:
shaped = (events
    .select(
        "event_id", "user_id", "timestamp", "amount",
        F.round(F.col("amount") * 1.21, 2).alias("with_vat"),
        F.when(F.col("amount") > 50, "big").otherwise("small").alias("size"),
        F.date_format("timestamp", "yyyy-MM-dd HH:mm").alias("ts_min"),
    ))

display(peek(shaped, "shaped"))

## The deterministic vs. non-deterministic split

This matters in streaming far more than in batch:

| Function | Deterministic? | Streaming behaviour |
|---|---|---|
| `current_timestamp()`, `current_date()` | Captured *per micro-batch*, not per row | Use freely |
| `rand()`, `randn()` (no seed) | Non-deterministic | Use freely, but mark UDFs that wrap them |
| `monotonically_increasing_id()` | Per partition counter, **not globally unique across batches** | Avoid for keys |
| `uuid()` | Truly random | Fine, but won't match across re-runs |
| Custom Python UDF reading a clock | Default: deterministic (wrong!) | Mark `.asNondeterministic()` |

In [ ]:
with_now = (events
    .withColumn("ingest_ts", F.current_timestamp())
    .withColumn("ingest_date", F.current_date())
    .withColumn("rid", F.expr("uuid()")))

display(peek(with_now.select("event_id", "ingest_ts", "ingest_date", "rid"), "with_now"))

## Aggregation needs a **watermark** — and only one of them per query

Most aggregate functions (`sum`, `count`, `count_distinct`, `collect_list`, `percentile_approx`) can be used in streaming, but only:

1. **Once per query** — multiple chained aggregations are unsupported.
2. **With a watermark** if the input is unbounded — otherwise state grows forever.
3. **With windowed grouping** for proper time-bucketed aggregates.

`approx_count_distinct` is *especially* useful in streaming because exact distinct keeps every key in state forever.

In [ ]:
windowed = (events
    .withWatermark("timestamp", "10 seconds")
    .groupBy(
        F.window("timestamp", "5 seconds"),
        "user_id",
    )
    .agg(
        F.count("*").alias("n"),
        F.sum("amount").alias("total"),
        F.approx_count_distinct("event_id").alias("approx_unique"),
    ))

display(peek(windowed.selectExpr("window.start AS w_start", "user_id", "n", "total", "approx_unique"),
             "windowed", seconds=8))

## Functions that are illegal on streaming DataFrames

| Banned | Why |
|---|---|
| `df.count()`, `df.collect()`, `df.show()` (as actions) | Would need the whole stream |
| `df.limit(n)` | Ambiguous on unbounded data |
| `df.sort()` / `orderBy()` (top-level) | Global sort impossible |
| `df.distinct()` without watermark | Unbounded state |
| Multiple chained aggregations | One per query only |
| Some subqueries | Catalyst can't plan them streaming-safely |

These are rejected at *analysis* time — the query never starts.

In [ ]:
from pyspark.errors import AnalysisException
try:
    bad = events.orderBy("amount").writeStream.format("memory").queryName("bad").start()
except (AnalysisException, Exception) as e:
    print("rejected:", type(e).__name__, str(e)[:120])

## UDFs in streams — same rules, sharper consequences

Plain Python UDFs work, but the per-row Python boundary is hit on **every micro-batch**, every trigger. A 5× cost in batch becomes a 5× cost on every trigger interval — meaning your streaming job either lags or needs more nodes.

Prefer:

1. **Built-ins** — same advice as batch.
2. **Pandas UDFs** — vectorized, often the only way to keep up with high throughput.
3. **`mapInPandas` / `applyInPandasWithState`** — for stateful per-key logic that doesn't fit standard aggregates.

In [ ]:
import pandas as pd

@F.pandas_udf("double")
def with_vat(s: pd.Series) -> pd.Series:
    return (s * 1.21).round(2)

priced = events.withColumn("price", with_vat("amount")).select("event_id", "amount", "price")
display(peek(priced, "priced"))

## Stateful Pandas UDFs — `applyInPandasWithState`

For per-key state that you'd otherwise hand-roll: sessionization, custom counters, leaky-bucket rate limits. The state is checkpointed automatically.

```python
def update(key, pdfs, state):
    # pdfs: iterator of pd.DataFrame for this key in this batch
    # state.exists / state.get / state.update / state.remove
    ...

events.withWatermark("timestamp", "30 seconds").groupBy("user_id") \
      .applyInPandasWithState(update, output_schema, state_schema, "Append", "EventTimeTimeout")
```

This is rich enough to deserve its own deep-dive (see `Streaming/08 - foreachBatch and Custom Sinks.ipynb` and `09 - Monitoring and Production Patterns.ipynb`).

## Watermarks and `current_timestamp` — a subtle gotcha

If you watermark on a column derived from `current_timestamp()` instead of an event-time column, you get **processing time**, not event time. Late data won't be late by your watermark; you've effectively turned watermarking off. Always watermark on the actual event-time column from your source.

## Takeaways

- Stateless functions (the bulk of `F.*`) work identically in batch and streaming.
- Streams allow **one** aggregation per query, and need a watermark for unbounded inputs.
- Prefer `approx_count_distinct` over `count_distinct` in streams — bounded state.
- `current_timestamp()` is captured per micro-batch — fine for ingestion timestamps, dangerous for event-time logic.
- `orderBy`, `limit`, multiple aggregations: rejected at analysis time.
- For heavy per-row logic in a stream, Pandas UDFs are usually the only path that keeps up with throughput.
- Stateful per-key logic → `applyInPandasWithState`.